In [33]:
from sklearn.datasets import fetch_openml, load_breast_cancer, load_iris
from ucimlrepo import fetch_ucirepo
import mlcroissant as mlc
import pandas as pd
 
name = 'adult'
 
lname = name.lower()
if lname == "adult":
    ds = fetch_openml(name="adult", version=2, as_frame=True, parser="auto")
    X = ds.data.copy()
    y = (ds.target.astype(str).str.contains(">")).astype(int).to_numpy()
elif lname == "mushroom":
    ds = fetch_openml(name="mushroom", version=1, as_frame=True, parser="auto")
    X = ds.data.copy()
    y = (ds.target.astype(str) == "p").astype(int).to_numpy()
elif lname == "breast_cancer":
    ds = load_breast_cancer(as_frame=True)
    X = ds.frame.copy()
    y = ds.target.copy()
elif lname == "iris":
    ds = load_iris(as_frame=True)
    X = ds.frame.drop(columns=["target"]).copy()
    y = ds.target.to_numpy()
elif lname == "mnist":
    ds = fetch_openml(name="mnist_784", version=1, as_frame=True, parser="auto")
    X = ds.data.copy()
    y = (
        ds.target.astype(int).to_numpy()
        if ds.target.dtype.kind in "iu"
        else ds.target.astype(str).astype(int).to_numpy()
    )
elif lname == "heart":
    X = pd.read_csv("data/real/heart.csv")
else:
    raise ValueError(f"Unknown dataset '{name}'")



In [34]:
X

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48837,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States
48838,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States
48839,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States
48840,22,Private,201490,HS-grad,9,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,target
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,0
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,1
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,1
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48837,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,0
48838,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,1
48839,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,0
48840,22,Private,201490,HS-grad,9,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States,0


In [27]:
df_train = df_train.sample(frac=1).reset_index(drop=True)
df_train.value_counts('over_threshold')[0]
number_clients = 10

label_dict = {}
for i in df_train['over_threshold'].unique():
    num_instances = df_train['over_threshold'].value_counts().get(i)
    exp_instances = num_instances // number_clients
    label_dict[i] = exp_instances 

print(label_dict)
clients_dfs = [pd.DataFrame() for _ in range(number_clients)]
for i in range(number_clients):
    for j in label_dict.keys():
        temp_df = df_train[df_train['over_threshold'] == j].sample(n=label_dict[j])
        clients_dfs[i] = pd.concat([clients_dfs[i], temp_df], ignore_index=True)
        df_train = df_train.drop(temp_df.index)
    
    print(df_train.shape[0])
    if df_train.shape[0] < 0.1*num_instances:
        print("Last small set")
        clients_dfs[i] = pd.concat([clients_dfs[i], df_train], ignore_index=True)

#display(clients_dfs[2].value_counts('over_threshold'))

{np.int64(0): np.int64(2786), np.int64(1): np.int64(876)}
32969
29307
25645
21983
18321
14659
10997
7335
3673
11
Last small set


In [31]:
for index, obj in enumerate(clients_dfs):
    clients_dfs[index] = torch.tensor(obj.values)
print(type(clients_dfs[0]))

<class 'torch.Tensor'>


In [16]:
import torch

array = [[0,1,2,0,0],[3,4,5,6,7]]
tensor = torch.tensor(array)
tensor.size(1)

5